# Figure 5 — Sweep results per $n$, curves by $\eta$

Loads the `(M, v_{\text{pop}})` samples persisted by `scripts/build_eta_pool.py`
(under `src/cache/eta_pool/<param_key>/eta{TT}/sample_NNNN/`), runs a uniform
bootstrap p-sweep on each, and plots `partition_agreement_M` vs $p$ — one
subplot per $n$, one curve per imbalance bin $\eta \in \{1, 5, 10, 15\}$.

Within an $\eta$ bin we average across the saved samples (mean ± std shading).
Sampling is **uniform** only; the metric is `partition_agreement_M` only.

## Pool status — samples cached per $(n, \eta)$

Quick read-only check of `src/cache/eta_pool/` for the param tuple this
notebook uses. Re-run this cell after `build_eta_pool*.py` finishes to see
fresh counts without running the full sweep below.

In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd()
while _ROOT.parent != _ROOT and not (_ROOT / "setup.py").exists():
    _ROOT = _ROOT.parent
_PROJECT_ROOT = _ROOT / "sub_sampled_fielder_vec"
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from src.utils.eta_pool_cache import (
    param_key as _param_key,
    list_completed_samples as _list_completed_samples,
    pool_root as _pool_root,
)

# Match Cell 1's param tuple — keep in sync if you change those.
_STATUS_MU         = 0.1
_STATUS_POP_SIZE   = 1.0
_STATUS_SEQ_LEN    = 10_000
_STATUS_TREE_MODEL = "kingman"
_STATUS_SEQ_MODEL  = "JC69"
_STATUS_ETA_TARGETS = [1, 5, 10, 15]
_STATUS_CACHE_ROOT = _PROJECT_ROOT / "src" / "cache"

_key_suffix = _param_key(
    n=0, seq_len=_STATUS_SEQ_LEN, mu=_STATUS_MU,
    tree_model=_STATUS_TREE_MODEL, pop_size=_STATUS_POP_SIZE,
    seq_model=_STATUS_SEQ_MODEL,
).split("_", 1)[1]

_root = _pool_root(_STATUS_CACHE_ROOT)
_rows = []
if _root.exists():
    for _d in sorted(_root.iterdir()):
        if not _d.is_dir() or not _d.name.endswith(_key_suffix):
            continue
        try:
            _n = int(_d.name.split("_", 1)[0][1:])
        except ValueError:
            continue
        _counts = [len(_list_completed_samples(_STATUS_CACHE_ROOT, _d.name, _e))
                   for _e in _STATUS_ETA_TARGETS]
        if any(_counts):
            _rows.append((_n, _counts))

_rows.sort(key=lambda r: r[0])
print(f"pool_root: {_root}")
print(f"params: L={_STATUS_SEQ_LEN}, mu={_STATUS_MU}, "
      f"tree={_STATUS_TREE_MODEL}, pop_size={_STATUS_POP_SIZE}, seq={_STATUS_SEQ_MODEL}")
print()
print(f"  {'n':>6}  " + "  ".join(f"eta={e:>2}" for e in _STATUS_ETA_TARGETS))
for _n, _cs in _rows:
    print(f"  {_n:>6}  " + "  ".join(f"{c:>6d}" for c in _cs))
if not _rows:
    print("  (no pool keys match these params)")


## Cell 1 — Configuration

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "setup.py").exists():
    ROOT = ROOT.parent
PROJECT_ROOT = ROOT / "sub_sampled_fielder_vec"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.eta_pool_cache import (
    param_key, bin_name, list_completed_samples, load_pool_entry, pool_root,
)
from src.runners.p_sweep_inner import bootstrap_p_sweep_simple
from src.utils.sweep_cache import compute_or_load_sweep

CACHE_ROOT = PROJECT_ROOT / "src" / "cache"

# Pool key parameters (must match build_eta_pool.py defaults).
MU         = 0.1
POP_SIZE   = 1.0
SEQ_LEN    = 10_000
TREE_MODEL = "kingman"
SEQ_MODEL  = "JC69"
ETA_TARGETS = [1, 5, 10, 15]

# Sweep parameters. Trim BOOTSTRAP_REPS / P_VALUES if you add bigger n.
P_VALUES                    = np.geomspace(1e-3, 1.0, 25).tolist()
BOOTSTRAP_REPS              = 10
SEED                        = 0
NUM_GAPS                    = 10
MIN_SPLIT                   = 1
MAX_SAMPLES_PER_BIN         = 10  # cap per-(n, eta) work; bins may have 20+ saved samples
EARLY_STOP_CONSECUTIVE_100  = 2  # stop sweep once this many consecutive p's hit 100%
# 'sign' matches the eta-bin definition (n1, n2 from sign of v_pop) and avoids
# partition_taxa picking degenerate (1, n-1) outlier splits.
PARTITION_METHOD            = "sign"
USE_SWEEP_CACHE             = True  # reload prior bootstrap_p_sweep_simple outputs from disk

print(f"pool_root: {pool_root(CACHE_ROOT)}")
print(f"p_values ({len(P_VALUES)}): {[f'{p:.3g}' for p in P_VALUES]}")
print(f"max_samples_per_bin: {MAX_SAMPLES_PER_BIN}")
print(f"early_stop_consecutive_100: {EARLY_STOP_CONSECUTIVE_100}")
print(f"partition_method: {PARTITION_METHOD}")
print(f"use_sweep_cache: {USE_SWEEP_CACHE}")

## Cell 2 — Discover available $n$ and saved samples

Walk `src/cache/eta_pool/`, parse the `n` from each `param_key` matching the
fixed `(L, μ, tree_model, pop_size, seq_model)` tuple, and report the sample
count per $\eta$ bin.

In [ ]:
# Build the param_key suffix that all matching n's share, and look for keys
# that start with 'n{N}' and end with the same suffix.
key_suffix = param_key(
    n=0, seq_len=SEQ_LEN, mu=MU, tree_model=TREE_MODEL,
    pop_size=POP_SIZE, seq_model=SEQ_MODEL,
).split("_", 1)[1]

ns = []
samples_by_n_eta = {}  # (n, eta_target) -> list[int] of completed sample idxs
for d in sorted(pool_root(CACHE_ROOT).iterdir()) if pool_root(CACHE_ROOT).exists() else []:
    if not d.is_dir() or not d.name.endswith(key_suffix):
        continue
    head = d.name.split("_", 1)[0]  # e.g. 'n0500'
    try:
        n = int(head[1:])
    except ValueError:
        continue
    key = d.name
    have_any = False
    for eta in ETA_TARGETS:
        idxs = list_completed_samples(CACHE_ROOT, key, eta)
        samples_by_n_eta[(n, eta)] = idxs
        have_any = have_any or bool(idxs)
    if have_any:
        ns.append(n)

ns = sorted(set(ns))
print(f"Found n values: {ns}")
print(f"\nSamples per (n, eta):")
print(f"  {'n':>6}  " + "  ".join(f"eta={e:>2}" for e in ETA_TARGETS))
for n in ns:
    counts = [len(samples_by_n_eta.get((n, e), [])) for e in ETA_TARGETS]
    print(f"  {n:>6}  " + "  ".join(f"{c:>6d}" for c in counts))

## Cell 3 — Run bootstrap p-sweeps

For each (n, η, sample), load the cached `(M, v_pop)` and call
`bootstrap_p_sweep_simple` (uniform sampling, partition_agreement_M).
Per-sample results are stored in `results[(n, eta)]` as a list of
agreement-curve arrays of length `len(P_VALUES)`.

In [ ]:
results = {}  # (n, eta) -> np.ndarray of shape (n_samples, len(P_VALUES))

for n in ns:
    key = param_key(
        n=n, seq_len=SEQ_LEN, mu=MU, tree_model=TREE_MODEL,
        pop_size=POP_SIZE, seq_model=SEQ_MODEL,
    )
    for eta in ETA_TARGETS:
        idxs = samples_by_n_eta.get((n, eta), [])[:MAX_SAMPLES_PER_BIN]
        if not idxs:
            results[(n, eta)] = np.empty((0, len(P_VALUES)))
            continue
        rows = []
        for idx in idxs:
            def _loader(_key=key, _eta=eta, _idx=idx):
                entry = load_pool_entry(CACHE_ROOT, _key, _eta, _idx)
                if entry is None:
                    return None
                M, v_pop, _meta = entry
                return M, v_pop

            outcome = compute_or_load_sweep(
                cache_root=CACHE_ROOT, param_key=key, eta_target=eta, idx=idx,
                M_loader=_loader,
                p_values=P_VALUES,
                bootstrap_reps=BOOTSTRAP_REPS,
                seed=SEED,
                num_gaps=NUM_GAPS,
                min_split=MIN_SPLIT,
                early_stop_consecutive_100=EARLY_STOP_CONSECUTIVE_100,
                partition_method=PARTITION_METHOD,
                use_cache=USE_SWEEP_CACHE,
            )
            if outcome is None:
                continue
            out, was_cached = outcome
            rows.append(out["partition_agreement_M"])
            tag = "cached" if was_cached else "computed"
            print(f"  n={n:>4}  eta={eta:>2}  sample={idx:>3}  "
                  f"split_ref={tuple(out['partition_split_ref'])}  "
                  f"agr@p_min={rows[-1][0]:5.1f}  agr@p_max={rows[-1][-1]:5.1f}  "
                  f"({tag})")
        results[(n, eta)] = np.asarray(rows) if rows else np.empty((0, len(P_VALUES)))

for (n, eta), arr in sorted(results.items()):
    print(f"  results[(n={n}, eta={eta})].shape = {arr.shape}")

## Cell 4 — Plot: one subplot per $n$, curves by $\eta$

Within each $\eta$ bin, the solid line is the mean across saved samples and
the shaded band is ±1 std. Bins with zero samples are skipped silently.

In [ ]:
if not ns:
    print("No matching pool keys found — nothing to plot.")
else:
    cmap = plt.get_cmap("viridis")
    eta_to_color = {
        e: cmap(i / max(1, len(ETA_TARGETS) - 1))
        for i, e in enumerate(ETA_TARGETS)
    }

    n_ns = len(ns)
    ncols = 2
    nrows = (n_ns + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(5 * ncols, 4.5 * nrows),
        sharey=True, squeeze=False,
    )
    axes_flat = axes.flatten()
    fig.suptitle(
        fr"Figure 5 — partition_agreement_M vs $p$ — "
        fr"tree={TREE_MODEL}, $\mu$={MU}, $L$={SEQ_LEN}, sampling=uniform, "
        fr"bootstrap_reps={BOOTSTRAP_REPS}"
    )

    for ax, n in zip(axes_flat, ns):
        for eta in ETA_TARGETS:
            arr = results.get((n, eta))
            if arr is None or arr.shape[0] == 0:
                continue
            mean = arr.mean(axis=0)
            std = arr.std(axis=0) if arr.shape[0] > 1 else np.zeros_like(mean)
            color = eta_to_color[eta]
            ax.plot(
                P_VALUES, mean, "-o", ms=4, lw=1.5, color=color,
                label=fr"$\eta$={eta}  ($k$={arr.shape[0]})",
            )
            if arr.shape[0] > 1:
                ax.fill_between(P_VALUES, mean - std, mean + std,
                                color=color, alpha=0.15, linewidth=0)

        ax.axhline(95.0, color="red", ls="--", lw=1, label="95% threshold")
        ax.set_xscale("log")
        ax.set_xlabel(r"Sampling probability $p$")
        ax.set_title(fr"$n$ = {n}")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(loc="lower right", fontsize=8)

    for ax in axes_flat[n_ns:]:
        ax.set_visible(False)

    for row in axes:
        row[0].set_ylabel("partition_agreement_M (%)")
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()

## Cell 5 — Plot 2 sweep: one sample per $(n, \eta)$, closest to bin target

For each $(n, \eta_{\text{target}})$, pick the cached sample whose measured
$\eta$ is closest to the target, run a single bootstrap p-sweep, and store
the result. No averaging across samples — one curve per $(n, \eta)$. This
mirrors the recovery-curves panel of `kingman_threshold_vs_theory.ipynb`.

In [ ]:
import json
from src.utils.eta_pool_cache import sample_dir

single_results = {}  # (n, eta_target) -> {"agreement", "eta_actual", "idx"}

for n in ns:
    key = param_key(
        n=n, seq_len=SEQ_LEN, mu=MU, tree_model=TREE_MODEL,
        pop_size=POP_SIZE, seq_model=SEQ_MODEL,
    )
    for eta_target in ETA_TARGETS:
        idxs = samples_by_n_eta.get((n, eta_target), [])
        if not idxs:
            continue

        # Read just metadata for each candidate to find the closest measured eta.
        candidates = []
        for idx in idxs:
            meta_path = sample_dir(CACHE_ROOT, key, eta_target, idx) / "metadata.json"
            if not meta_path.exists():
                continue
            with open(meta_path, "r") as f:
                meta = json.load(f)
            candidates.append((idx, float(meta["eta"])))
        if not candidates:
            continue

        idx_star, eta_actual = min(
            candidates, key=lambda t: abs(t[1] - float(eta_target))
        )

        def _loader(_key=key, _eta=eta_target, _idx=idx_star):
            entry = load_pool_entry(CACHE_ROOT, _key, _eta, _idx)
            if entry is None:
                return None
            M, v_pop, _meta = entry
            return M, v_pop

        outcome = compute_or_load_sweep(
            cache_root=CACHE_ROOT, param_key=key, eta_target=eta_target, idx=idx_star,
            M_loader=_loader,
            p_values=P_VALUES,
            bootstrap_reps=BOOTSTRAP_REPS,
            seed=SEED,
            num_gaps=NUM_GAPS,
            min_split=MIN_SPLIT,
            early_stop_consecutive_100=EARLY_STOP_CONSECUTIVE_100,
            partition_method=PARTITION_METHOD,
            use_cache=USE_SWEEP_CACHE,
        )
        if outcome is None:
            continue
        out, was_cached = outcome
        agr = np.asarray(out["partition_agreement_M"], dtype=float)
        single_results[(n, eta_target)] = {
            "agreement": agr,
            "eta_actual": eta_actual,
            "idx": idx_star,
        }
        tag = "cached" if was_cached else "computed"
        print(
            f"  n={n:>4}  eta_target={eta_target:>2}  idx={idx_star:>3}  "
            f"eta_actual={eta_actual:5.2f}  "
            f"agr@p_min={agr[0]:5.1f}  agr@p_max={agr[-1]:5.1f}  ({tag})"
        )

print(f"\nselected {len(single_results)} (n, eta_target) curves")

## Cell 6 — Plot 2: one subplot per $\eta$, curves by $n$

Mirrors the recovery-curves panel of `kingman_threshold_vs_theory.ipynb`:
each subplot fixes a $\eta$ bin, and each curve is a different $n$. Only one
sample per curve (closest measured $\eta$ to the bin target).

In [ ]:
if not single_results:
    print("No single-sample sweeps available — nothing to plot.")
else:
    cmap = plt.get_cmap("viridis")
    n_to_color = {
        n: cmap(i / max(1, len(ns) - 1))
        for i, n in enumerate(ns)
    }

    n_etas = len(ETA_TARGETS)
    ncols = 2
    nrows = (n_etas + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(5 * ncols, 4.5 * nrows),
        sharey=True, squeeze=False,
    )
    axes_flat = axes.flatten()
    fig.suptitle(
        fr"Figure 5b — partition_agreement_M vs $p$ — "
        fr"tree={TREE_MODEL}, $\mu$={MU}, $L$={SEQ_LEN}, sampling=uniform, "
        fr"partition={PARTITION_METHOD}, bootstrap_reps={BOOTSTRAP_REPS}"
    )

    for ax, eta_target in zip(axes_flat, ETA_TARGETS):
        for n in ns:
            entry = single_results.get((n, eta_target))
            if entry is None:
                continue
            ax.plot(
                P_VALUES, entry["agreement"], "-o", ms=4, lw=1.5,
                color=n_to_color[n],
                label=fr"$n$={n}, $\eta$={entry['eta_actual']:.2f}",
            )

        ax.axhline(95.0, color="red", ls="--", lw=1, label="95% threshold")
        ax.set_xscale("log")
        ax.set_xlabel(r"Sampling probability $p$")
        ax.set_title(fr"$\eta \approx {eta_target}$")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(loc="lower right", fontsize=8)

    for ax in axes_flat[n_etas:]:
        ax.set_visible(False)

    for row in axes:
        row[0].set_ylabel("partition_agreement_M (%)")
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()

## Cell 7 — Plot 3: critical $p^*$ vs $n$, one curve per $\eta$

For each $(n, \eta)$, take the mean `partition_agreement_M` across saved samples
(from Cell 3's `results`), and define $p^*(n, \eta)$ as the smallest $p$ in the grid
at which the mean agreement reaches the 95% recovery threshold. Plot $p^*$ vs $n$
on log–log axes with one curve per $\eta$ — the figure-3 right-panel summary, but
collapsed into a single plot.

In [ ]:
from analysis.comparison.phase_transition_utils import find_discrete_threshold

RECOVERY_THRESHOLD_PCT = 95.0  # results[*] is in percent (0..100)

# Semantic palette: gray = balanced reference (eta=1), blue→orange→red = increasing imbalance.
ETA_COLORS = {
    1:  "#4b5563",
    5:  "#1d4ed8",
    10: "#ea580c",
    15: "#b91c1c",
}

p_arr = np.asarray(P_VALUES, dtype=float)
p_star_by_eta = {}  # eta -> list of (n, p_star) with finite p_star

for eta in ETA_TARGETS:
    rows = []
    for n in ns:
        arr = results.get((n, eta))
        if arr is None or arr.shape[0] == 0:
            continue
        mean_curve = arr.mean(axis=0)
        p_star = find_discrete_threshold(p_arr, mean_curve, threshold=RECOVERY_THRESHOLD_PCT)
        if p_star is None or not np.isfinite(p_star):
            continue
        rows.append((n, float(p_star)))
    p_star_by_eta[eta] = rows
    print(f"  eta={eta:>2}: " + ", ".join(f"n={n}→p*={p:.4g}" for n, p in rows))

fig, ax = plt.subplots(figsize=(7, 7))
for eta in ETA_TARGETS:
    rows = p_star_by_eta.get(eta, [])
    if not rows:
        continue
    xs = np.array([r[0] for r in rows], dtype=float)
    ys = np.array([r[1] for r in rows], dtype=float)
    ax.plot(
        xs, ys, "-o", ms=6, lw=2,
        color=ETA_COLORS.get(eta, "#000000"),
        label=fr"$\eta$ = {eta}",
    )

# log(n)/n reference (balanced-binary scaling), normalized to the eta=1 curve at its smallest n.
ref_rows = p_star_by_eta.get(1, [])
if ref_rows:
    n_ref, p_ref = ref_rows[0]
    n_grid = np.array(ns, dtype=float)
    theory = p_ref * (np.log(n_grid) / n_grid) / (np.log(n_ref) / n_ref)
    ax.plot(n_grid, theory, ":", color="#4b5563", lw=1.5,
            label=r"$\log n / n$ reference (anchored at $\eta=1$)")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"Number of taxa $n$")
ax.set_ylabel(r"Critical sampling probability $p^*$  (smallest $p$ with mean partition_agreement_M $\geq 95\%$)")
ax.set_title(
    fr"$p^*(n)$ for kingman tree, $\mu$={MU}, $L$={SEQ_LEN}, uniform sampling, "
    fr"bootstrap_reps={BOOTSTRAP_REPS}"
)
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="lower left", fontsize=10)
fig.tight_layout()
plt.show()